In [6]:
from typing import Optional

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


# -------------------------
# 1. تعریف Schema
# -------------------------

class Employee(BaseModel):

    name: str = Field(
        description="نام و نام خانوادگی کارمند"
    )

    age: int = Field(
        description="سن کارمند"
    )

    job_title: str = Field(
        description="عنوان شغلی کارمند"
    )

    department: str = Field(
        description="واحد یا دپارتمان محل کار"
    )

    years_of_experience: int = Field(
        description="تعداد سال سابقه کاری"
    )

    is_manager: bool = Field(
        description="آیا فرد مدیر است یا خیر"
    )

    salary: Optional[int] = Field(
        default=None,
        description="حقوق ماهانه در صورت وجود"
    )


# -------------------------
# 2. ساخت Parser
# -------------------------

parser = JsonOutputParser(
    pydantic_object=Employee
)


# -------------------------
# 3. ساخت Prompt
# -------------------------

prompt = ChatPromptTemplate.from_template("""
اطلاعات کارمند را از متن زیر استخراج کن.

متن:
{name}

قوانین:
- فقط JSON معتبر برگردان.
- هیچ توضیحی خارج از JSON ننویس.
- اگر مقدار salary وجود نداشت، مقدار null قرار بده.
- is_manager باید true یا false باشد.
- years_of_experience و age باید عدد باشند.

Schema:
{format_instructions}
""")


# -------------------------
# 4. Model
# -------------------------

model = init_chat_model(
    "qwen3:1.7b",
    model_provider="ollama",
    temperature=0
)


# -------------------------
# 5. Chain
# -------------------------

chain = prompt | model | parser


# -------------------------
# 6. اجرا
# -------------------------

result = chain.invoke({
    "name": """
    فرزاد احمدی 35 ساله است و به عنوان
    برنامه نویس ارشد در واحد فناوری اطلاعات کار می‌کند.
    او 10 سال سابقه کاری دارد و مدیر نیست.
    حقوق ماهانه او 85000000 تومان است.
    """,

    "format_instructions": parser.get_format_instructions()
})


# -------------------------
# 7. نتیجه
# -------------------------

print(result)
print(type(result))

{'name': 'فرزاد احمدی', 'age': 35, 'job_title': 'برنامه نویس ارشد', 'department': 'واحد فناوری اطلاعات', 'years_of_experience': 10, 'is_manager': False, 'salary': 85000000}
<class 'dict'>
